Gold layer: COVID-19 local outbreak spike detection

For each geography, compares the most recent 7-day case total against the
7 days before that, and flags a spike if the percentage change exceeds the
threshold. Keeps the raw counts alongside the percentage so small-number
swings (1 to 2 cases) can be told apart from genuinely concerning ones
(100 to 200 cases) at a glance, rather than the percentage hiding context.

In [0]:
# Imports and config
from datetime import timedelta
from pyspark.sql.functions import col, sum as spark_sum, when, max as spark_max, lit
 
silver_path = "abfss://silver@ukhsadev2026.dfs.core.windows.net/ukhsa/covid19_cases_by_day"
gold_path = "abfss://gold@ukhsadev2026.dfs.core.windows.net/ukhsa/covid19_outbreak_alerts"

# Threshold value
# Change for different sensitivity
THRESHOLD_PERCENT = 25.0

In [0]:
# Read Silver and calculate our comparison windows (current week and previous week)

# Reads in Silver
silver_df = spark.read.format("delta").load(silver_path)
 
# Taking the most recent date in the data set (might not be today)
latest_date = silver_df.agg(spark_max("date")).collect()[0][0]
 
# Calculate the comparison windows based on the most recent date
current_week_start = latest_date - timedelta(days=6)
previous_week_end = latest_date - timedelta(days=7)
previous_week_start = latest_date - timedelta(days=13)

In [0]:
# Sum cases per geography for each window, then join the two together
 
# Current week: latest_date - 6 .. latest_date (7 days inclusive)
current_week_df = (
    silver_df.filter((col("date") >= current_week_start) & (col("date") <= latest_date))
    .groupBy("geography", "geography_code")
    .agg(spark_sum("metric_value").alias("current_week_cases"))
)
 
# Previous week: the 7 days immediately before the "current_week"
previous_week_df = (
    silver_df.filter((col("date") >= previous_week_start) & (col("date") <= previous_week_end))
    .groupBy("geography", "geography_code")
    .agg(spark_sum("metric_value").alias("previous_week_cases"))
)
 

# Joining the current and previous week 
# Using inner join to only look at areas with enough data
combined_df = current_week_df.join(
    previous_week_df, on=["geography", "geography_code"], how="inner"
)
 

In [0]:
# Calculate percent change and apply the spike threshold

# Creating the gold dataframe
gold_df = (
    combined_df.withColumn(
        "percent_change",
        when(
            col("previous_week_cases") > 0,
            ((col("current_week_cases") - col("previous_week_cases")) / col("previous_week_cases")) * 100,
        ).otherwise(None),  # avoid divide-by-zero, previous week had 0 cases
    )
    .withColumn(
        "alert_status",
        when(col("percent_change") > THRESHOLD_PERCENT, "SPIKE").otherwise("BASELINE"),
    )
    .withColumn("data_as_of", lit(latest_date))
    .select(
        "geography",
        "geography_code",
        "data_as_of",
        "current_week_cases",
        "previous_week_cases",
        "percent_change",
        "alert_status",
    )
)
 
# data_as_of is the most recent date UKHSA had published data for when this
# Gold run happened. The three other date boundaries are always fixed
# offsets from it, 6, 7, and 13 days back, not stored since they'd repeat
# identically on every row. Recompute when needed:
#   current_week_start  = data_as_of - 6 days
#   previous_week_end   = data_as_of - 7 days
#   previous_week_start = data_as_of - 13 days

In [0]:
#Write to Gold 

# Create Gold delta table and save it to the external gold layer storage location (azure) 
gold_df.write.format("delta").mode("overwrite").save(gold_path)

# Confirm that the delta table has been saved in the Gold layer's storage
print(f"Gold write complete: {gold_df.count()} geographies")

# Display the dataframe (delta table) from the highest change to the least change.
gold_df.orderBy(col("percent_change").desc()).show(20, truncate=False)